In [41]:
import os
import time
import joblib
import logging
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.preprocessing import MaxAbsScaler

logging.basicConfig(level=logging.INFO)

In [42]:
def load_data(input_path):
    logging.info("Loading data...")
    df = pd.read_csv(os.path.join(input_path, 'telephish.csv'),
                     low_memory=False)
    return df


In [43]:
def preprocess_data(df):
    logging.info("Preprocessing data...")

    # Ensure boolean features are integers
    bool_features = ['is_forwarded', 'is_bot', 'has_media', 'has_text',
                     'is_reply', 'language_match', 'username_matches_language']
    for feature in bool_features:
        df[feature] = df[feature].astype(int)

    # Convert message time to categorical parts of the day
    df['time_of_day'] = df.apply(categorize_time, axis=1)
    df = pd.get_dummies(df, columns=['time_of_day'])

    return df



In [44]:
def categorize_time(row):
    """Categorize the time of day based on the message timestamp."""
    hour = pd.to_datetime(row['time']).hour
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Night'
    else:
        return 'Midnight'


In [45]:
def prepare_features(df, features):
    logging.info("Preparing features and target variables...")

    initial_row_count = len(df)
    df_dedup = df.drop_duplicates(subset=features)
    dedup_row_count = len(df_dedup)
    rows_dropped = initial_row_count - dedup_row_count
    logging.info(f"Number of duplicate rows dropped: {rows_dropped}")

    X = df_dedup[features]
    y = df_dedup['is_phishing']
    sample_ids = df_dedup["sample_ids"]
    category = df_dedup["category"]

    missing_in_X = X.isnull().sum()
    missing_columns_X = missing_in_X[missing_in_X > 0].index.tolist()
    if missing_columns_X:
        logging.warning(f"Columns with missing values in X: {missing_columns_X}")
        df_clean = df_dedup.dropna(subset=missing_columns_X).reset_index(drop=True)
        clean_row_count = len(df_clean)
        rows_dropped_missing = dedup_row_count - clean_row_count
        logging.info(f"Number of rows dropped due to missing values: {rows_dropped_missing}")
        logging.info(f"Number of rows after dropping missing values: {clean_row_count}")
        X = df_clean[features]
        y = df_clean['is_phishing']
        sample_ids = df_clean["sample_ids"]
        category = df_clean["category"]

    if y.isnull().any():
        missing_rows_y = y[y.isnull()].index.tolist()
        logging.warning(f"Target vector 'y' has missing values in rows: {missing_rows_y}")

    return X, y, sample_ids, category


In [46]:
def build_and_evaluate_models_per_category(X, y, sample_ids, category, path_prefix):

    numerical_features = [
        'message_length', 
        'url_count', 
        'total_messages', 
        'unique_users_per_group_message', 
        'messages_repeat_by_user',
        'formatted_text_count'
    ]
    
    param_grid = {
        'n_estimators': [100, 200, 500],
        'max_depth': [None, 5, 10, 20],
        'min_samples_split': [2, 5, 10],
        'class_weight': ['balanced', 'balanced_subsample']
    }
    
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    
    results = {}
    best_params_dict = {}  # key: category, value: list of best param dicts from each outer fold
    agg_predictions = {}   # key: category, value: aggregated predictions dataframe

    unique_cats = sorted(category.dropna().unique())
    for cat in unique_cats:
        results[cat] = {
            'precision': [],
            'recall': [],
            'f1': [],
            'precision_neg': [],
            'recall_neg': [],
            'f1_neg': [],
            'sensitivity': [],
            'specificity': [],
            'gmean': []
        }
        best_params_dict[cat] = []
        agg_predictions[cat] = []  # will store dataframes with prediction
    
    fold_idx = 0
    for outer_train_index, outer_test_index in outer_cv.split(X, y):
        fold_idx += 1
        logging.info(f"Outer Fold {fold_idx} starting...")
        X_outer_train = X.iloc[outer_train_index].reset_index(drop=True)
        y_outer_train = y.iloc[outer_train_index].reset_index(drop=True)
        sample_ids_outer_train = sample_ids.iloc[outer_train_index].reset_index(drop=True)
        category_outer_train = category.iloc[outer_train_index].reset_index(drop=True)
        
        X_outer_test = X.iloc[outer_test_index].reset_index(drop=True)
        y_outer_test = y.iloc[outer_test_index].reset_index(drop=True)
        sample_ids_outer_test = sample_ids.iloc[outer_test_index].reset_index(drop=True)
        category_outer_test = category.iloc[outer_test_index].reset_index(drop=True)
        
        for cat in unique_cats:
            train_mask = (category_outer_train == cat)
            X_train_cat = X_outer_train[train_mask].reset_index(drop=True)
            y_train_cat = y_outer_train[train_mask].reset_index(drop=True)
            
            test_mask = (category_outer_test == cat)
            X_test_cat = X_outer_test[test_mask].reset_index(drop=True)
            y_test_cat = y_outer_test[test_mask].reset_index(drop=True)
            sample_ids_test_cat = sample_ids_outer_test[test_mask].reset_index(drop=True)
            
            if len(X_train_cat) < 10:
                logging.warning(f"Not enough training samples for category {cat} in outer fold {fold_idx}. Skipping.")
                continue
            
            scaler = MaxAbsScaler()
            scaler.fit(X_train_cat[numerical_features])
            X_train_cat_scaled = X_train_cat.copy()
            X_test_cat_scaled = X_test_cat.copy()
            X_train_cat_scaled[numerical_features] = scaler.transform(X_train_cat[numerical_features])
            if not X_test_cat_scaled.empty:
                X_test_cat_scaled[numerical_features] = scaler.transform(X_test_cat[numerical_features])
            
            inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
            model_to_tune = RandomForestClassifier(random_state=0)
            grid_search = GridSearchCV(estimator=model_to_tune, param_grid=param_grid,
                                       cv=inner_cv, n_jobs=-1, scoring='f1_macro')
            grid_search.fit(X_train_cat_scaled, y_train_cat)
            best_params = grid_search.best_params_
            best_params_dict[cat].append(best_params)
            
            model_cat = RandomForestClassifier(**best_params, random_state=0)
            model_cat.fit(X_train_cat_scaled, y_train_cat)
            
            if not X_test_cat_scaled.empty:
                y_pred_cat = model_cat.predict(X_test_cat_scaled)
                cm = confusion_matrix(y_test_cat, y_pred_cat)
                
                TN = cm[0, 0] if cm.shape[0] > 1 and cm.shape[1] > 1 else 0
                FP = cm[0, 1] if cm.shape[1] > 1 else 0
                FN = cm[1, 0] if cm.shape[0] > 1 else 0
                TP = cm[1, 1] if cm.shape[0] > 1 and cm.shape[1] > 1 else 0
                sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0
                specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0
                gmean = np.sqrt(sensitivity * specificity)
                
                # Compute metrics for both classes (negative: label 0, positive: label 1)
                precision_arr, recall_arr, f1_arr, _ = precision_recall_fscore_support(
                    y_test_cat, y_pred_cat, labels=[0, 1], zero_division=0
                )
                precision_neg, precision_pos = precision_arr
                recall_neg, recall_pos = recall_arr
                f1_neg, f1_pos = f1_arr

                results[cat]['precision'].append(precision_pos)
                results[cat]['recall'].append(recall_pos)
                results[cat]['f1'].append(f1_pos)
                results[cat]['precision_neg'].append(precision_neg)
                results[cat]['recall_neg'].append(recall_neg)
                results[cat]['f1_neg'].append(f1_neg)
                results[cat]['sensitivity'].append(recall_pos)
                results[cat]['specificity'].append(recall_neg)
                results[cat]['gmean'].append(gmean)
                
                # Save predictions for this fold and category.
                fold_data = X_test_cat.copy()
                fold_data['sample_ids'] = sample_ids_test_cat
                fold_data['actual'] = y_test_cat
                fold_data['predicted'] = y_pred_cat
                fold_data['category'] = cat
                fold_file = os.path.join(path_prefix, f"{cat}_fold{fold_idx}_predictions.csv")
                fold_data.to_csv(fold_file, index=False)
                
                # Also aggregate predictions to later save a complete file.
                agg_predictions[cat].append(fold_data)
                
                print(f"Outer Fold {fold_idx} - Category {cat}:")
                print(pd.DataFrame(cm,
                                   index=['Non-Malicious (Negative)', 'Malicious (Positive)'],
                                   columns=['Predicted Non-Malicious', 'Predicted Malicious']))
                print(f"   Positive Precision: {precision_pos:.3f}, Positive Recall: {recall_pos:.3f}, Positive F1-score: {f1_pos:.3f}")
                print(f"   Negative Precision: {precision_neg:.3f}, Negative Recall: {recall_neg:.3f}, Negative F1-score: {f1_neg:.3f}")
                print(f"   Sensitivity (TPR): {sensitivity:.3f}")
                print(f"   Specificity (TNR): {specificity:.3f}")
                print(f"   G-mean: {gmean:.3f}\n")
            else:
                logging.info(f"No test samples for category {cat} in outer fold {fold_idx}.")
    
    # Print aggregated metrics (average and standard deviation) for each category
    print("\nAggregated Metrics per Category (across outer folds):")
    for cat in unique_cats:
        prec_list = np.array(results[cat]['precision'])
        rec_list = np.array(results[cat]['recall'])
        f1_list = np.array(results[cat]['f1'])
        prec_neg_list = np.array(results[cat]['precision_neg'])
        rec_neg_list = np.array(results[cat]['recall_neg'])
        f1_neg_list = np.array(results[cat]['f1_neg'])
        sens_list = np.array(results[cat]['sensitivity'])
        spec_list = np.array(results[cat]['specificity'])
        gmean_list = np.array(results[cat]['gmean'])
        
        if prec_list.size > 0:
            print(f"\nCategory: {cat}")
            print(f"   Positive Precision: {prec_list.mean():.3f} ± {prec_list.std():.3f}")
            print(f"   Positive Recall:    {rec_list.mean():.3f} ± {rec_list.std():.3f}")
            print(f"   Positive F1-score:  {f1_list.mean():.3f} ± {f1_list.std():.3f}")
            print(f"   Negative Precision: {prec_neg_list.mean():.3f} ± {prec_neg_list.std():.3f}")
            print(f"   Negative Recall:    {rec_neg_list.mean():.3f} ± {rec_neg_list.std():.3f}")
            print(f"   Negative F1-score:  {f1_neg_list.mean():.3f} ± {f1_neg_list.std():.3f}")
            print(f"   Sensitivity (TPR):  {sens_list.mean():.3f} ± {sens_list.std():.3f}")
            print(f"   Specificity (TNR):  {spec_list.mean():.3f} ± {spec_list.std():.3f}")
            print(f"   G-mean:             {gmean_list.mean():.3f} ± {gmean_list.std():.3f}")
        else:
            print(f"\nCategory: {cat} - No evaluation metrics available.")
    
    # Aggregate best parameters per category (take mode across outer folds)
    aggregated_best_params = {}
    param_types = {
        'n_estimators': int,
        'max_depth': lambda x: int(x) if x is not None else None,
        'min_samples_split': int,
        'class_weight': lambda x: x
    }
    
    for cat, params_list in best_params_dict.items():
        if params_list:
            best_params_df = pd.DataFrame(params_list)
            aggregated_params = best_params_df.mode().iloc[0].to_dict()
            # Convert parameters to correct types.
            for param, conv in param_types.items():
                if param in aggregated_params:
                    aggregated_params[param] = conv(aggregated_params[param])
            aggregated_best_params[cat] = aggregated_params
            logging.info(f"Aggregated best parameters for category {cat}: {aggregated_params}")
        else:
            logging.info(f"No best parameters collected for category {cat}.")
    
    best_params_file = os.path.join(path_prefix, "aggregated_best_params_per_category.csv")
    pd.DataFrame(aggregated_best_params).to_csv(best_params_file, index=True)
    
    # Save aggregated predictions per category.
    for cat in unique_cats:
        if agg_predictions[cat]:
            all_preds = pd.concat(agg_predictions[cat], axis=0).reset_index(drop=True)
            agg_pred_file = os.path.join(path_prefix, f"{cat}_aggregated_predictions.csv")
            all_preds.to_csv(agg_pred_file, index=False)
    
    return aggregated_best_params


In [47]:
def train_final_models_per_category(X, y, sample_ids, category, path_prefix, aggregated_best_params):

    numerical_features = [
        'message_length', 
        'url_count', 
        'total_messages', 
        'unique_users_per_group_message', 
        'messages_repeat_by_user',
        'formatted_text_count'
    ]
    
    for cat, best_params in aggregated_best_params.items():
        mask = (category == cat)
        X_cat = X[mask].reset_index(drop=True)
        y_cat = y[mask].reset_index(drop=True)
        sample_ids_cat = sample_ids[mask].reset_index(drop=True)
        
        if len(X_cat) == 0:
            logging.info(f"No samples for category {cat}. Skipping final model training.")
            continue
        
        scaler = MaxAbsScaler()
        scaler.fit(X_cat[numerical_features])
        X_cat_scaled = X_cat.copy()
        X_cat_scaled[numerical_features] = scaler.transform(X_cat[numerical_features])
        
        model_final = RandomForestClassifier(**best_params, random_state=0)
        start_time = time.time()
        model_final.fit(X_cat_scaled, y_cat)
        duration = time.time() - start_time
        hours, rem = divmod(duration, 3600)
        minutes, seconds = divmod(rem, 60)
        logging.info(f"Final model for category {cat} trained in {int(hours)}h {int(minutes)}m {seconds:.2f}s")
        
        model_file = os.path.join(path_prefix, f"{cat}_final_random_forest_model.pkl")
        scaler_file = os.path.join(path_prefix, f"{cat}_scaler.pkl")
        joblib.dump(model_final, model_file)
        joblib.dump(scaler, scaler_file)
        
        params_file = os.path.join(path_prefix, f"{cat}_final_model_params.txt")
        with open(params_file, 'w') as f:
            for param, value in model_final.get_params().items():
                f.write(f"{param}: {value}\n")
        logging.info(f"Final model and scaler for category {cat} saved.")

In [48]:
def models_per_category(df, path_prefix):
    selected_features = [
        'is_bot', 'message_length', 'has_media', 'unique_users_per_group_message',
        'is_reply', 'url_count', 'total_messages', 'normalized_days_until_first_post',
        'language_match', 'messages_repeat_by_user', 'username_matches_language', 'formatted_text_count'
    ]
    X, y, sample_ids, category = prepare_features(df, selected_features)
    
    logging.info(f"Unique target labels: {np.unique(y, return_counts=True)}")
    logging.info(f"Unique categories: {np.unique(category)}")
    
    os.makedirs(path_prefix, exist_ok=True)
    
    aggregated_best_params = build_and_evaluate_models_per_category(X, y, sample_ids, category, path_prefix)
    
    train_final_models_per_category(X, y, sample_ids, category, path_prefix, aggregated_best_params)


In [49]:
def main():
    save_path = 'results/'
    input_path = '../../../data/'
    
    os.makedirs(save_path, exist_ok=True)
    df = load_data(input_path)
    df = preprocess_data(df)
    
    models_per_category(df, save_path)

In [50]:
if __name__ == '__main__':
    main()

INFO:root:Loading data...
INFO:root:Preprocessing data...
INFO:root:Preparing features and target variables...
INFO:root:Number of duplicate rows dropped: 7315
INFO:root:Number of rows dropped due to missing values: 1017
INFO:root:Number of rows after dropping missing values: 49877
INFO:root:Unique target labels: (array([False,  True]), array([48880,   997]))
INFO:root:Unique categories: ['Crypto' 'Darknet' 'Games']
INFO:root:Outer Fold 1 starting...


Outer Fold 1 - Category Crypto:
                          Predicted Non-Malicious  Predicted Malicious
Non-Malicious (Negative)                     5715                    5
Malicious (Positive)                           23                   20
   Positive Precision: 0.800, Positive Recall: 0.465, Positive F1-score: 0.588
   Negative Precision: 0.996, Negative Recall: 0.999, Negative F1-score: 0.998
   Sensitivity (TPR): 0.465
   Specificity (TNR): 0.999
   G-mean: 0.682
Outer Fold 1 - Category Darknet:
                          Predicted Non-Malicious  Predicted Malicious
Non-Malicious (Negative)                      554                    7
Malicious (Positive)                            7                    4
   Positive Precision: 0.364, Positive Recall: 0.364, Positive F1-score: 0.364
   Negative Precision: 0.988, Negative Recall: 0.988, Negative F1-score: 0.988
   Sensitivity (TPR): 0.364
   Specificity (TNR): 0.988
   G-mean: 0.599


INFO:root:Outer Fold 2 starting...


Outer Fold 1 - Category Games:
                          Predicted Non-Malicious  Predicted Malicious
Non-Malicious (Negative)                     3476                   19
Malicious (Positive)                           30                  116
   Positive Precision: 0.859, Positive Recall: 0.795, Positive F1-score: 0.826
   Negative Precision: 0.991, Negative Recall: 0.995, Negative F1-score: 0.993
   Sensitivity (TPR): 0.795
   Specificity (TNR): 0.995
   G-mean: 0.889
Outer Fold 2 - Category Crypto:
                          Predicted Non-Malicious  Predicted Malicious
Non-Malicious (Negative)                     5800                    7
Malicious (Positive)                           26                   13
   Positive Precision: 0.650, Positive Recall: 0.333, Positive F1-score: 0.441
   Negative Precision: 0.996, Negative Recall: 0.999, Negative F1-score: 0.997
   Sensitivity (TPR): 0.333
   Specificity (TNR): 0.999
   G-mean: 0.577
Outer Fold 2 - Category Darknet:
                

INFO:root:Outer Fold 3 starting...


...

INFO:root:Outer Fold 5 starting...


Outer Fold 4 - Category Games:
                          Predicted Non-Malicious  Predicted Malicious
Non-Malicious (Negative)                     3522                   16
Malicious (Positive)                           31                  110
   Positive Precision: 0.873, Positive Recall: 0.780, Positive F1-score: 0.824
   Negative Precision: 0.991, Negative Recall: 0.995, Negative F1-score: 0.993
   Sensitivity (TPR): 0.780
   Specificity (TNR): 0.995
   G-mean: 0.881
Outer Fold 5 - Category Crypto:
                          Predicted Non-Malicious  Predicted Malicious
Non-Malicious (Negative)                     5788                    6
Malicious (Positive)                           27                   17
   Positive Precision: 0.739, Positive Recall: 0.386, Positive F1-score: 0.507
   Negative Precision: 0.995, Negative Recall: 0.999, Negative F1-score: 0.997
   Sensitivity (TPR): 0.386
   Specificity (TNR): 0.999
   G-mean: 0.621
Outer Fold 5 - Category Darknet:
                

INFO:root:Aggregated best parameters for category Crypto: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 10, 'n_estimators': 500}
INFO:root:Aggregated best parameters for category Darknet: {'class_weight': 'balanced_subsample', 'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 500}
INFO:root:Aggregated best parameters for category Games: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 200}


Outer Fold 5 - Category Games:
                          Predicted Non-Malicious  Predicted Malicious
Non-Malicious (Negative)                     3394                   28
Malicious (Positive)                           23                  113
   Positive Precision: 0.801, Positive Recall: 0.831, Positive F1-score: 0.816
   Negative Precision: 0.993, Negative Recall: 0.992, Negative F1-score: 0.993
   Sensitivity (TPR): 0.831
   Specificity (TNR): 0.992
   G-mean: 0.908


Aggregated Metrics per Category (across outer folds):

Category: Crypto
   Positive Precision: 0.701 ± 0.066
   Positive Recall:    0.376 ± 0.063
   Positive F1-score:  0.489 ± 0.069
   Negative Precision: 0.995 ± 0.000
   Negative Recall:    0.999 ± 0.000
   Negative F1-score:  0.997 ± 0.000
   Sensitivity (TPR):  0.376 ± 0.063
   Specificity (TNR):  0.999 ± 0.000
   G-mean:             0.611 ± 0.052

Category: Darknet
   Positive Precision: 0.351 ± 0.079
   Positive Recall:    0.472 ± 0.138
   Positive F1-score:  0.

INFO:root:Final model for category Crypto trained in 0h 0m 4.27s
INFO:root:Final model and scaler for category Crypto saved.
INFO:root:Final model for category Darknet trained in 0h 0m 0.67s
INFO:root:Final model and scaler for category Darknet saved.
INFO:root:Final model for category Games trained in 0h 0m 1.01s
INFO:root:Final model and scaler for category Games saved.
